# RAPIDS cuDF's pandas accelerator mode (cudf.pandas)

<img src="https://raw.githubusercontent.com/rapidsai-community/tutorial/refs/heads/main/images/cudf-pandas-exec-flow.png" style="float: right; margin-left: 5px; width: 250px;">

In the previous notebook we learn how `cuDF` help us leverage the GPU while keeping a familiar pandas API. But `cuDF` 
also has a zero-code-change feature that can help you get better performance from your existing pandas code.

`cuDF` provides a pandas accelerator mode (`cudf.pandas`), allowing you to bring accelerated computing to your pandas 
workflows without requiring any code change.


## Why should I use `cudf.pandas`?

- Requires no changes to existing pandas code. Just
    - `%load_ext cudf.pandas`
    - `$ python –m cudf.pandas <script.py>`
    - If you have `jupyterlab-nvdashboard >=0.14`, use the GPU Accel toggle 
- 100% of the pandas API
- Accelerates workflows up to [150x using the GPU](https://developer.nvidia.com/blog/rapids-cudf-accelerates-pandas-nearly-150x-with-zero-code-changes/)
- Compatible with code that uses third-party libraries
- Falls back to using pandas on the CPU for unsupported functions and methods

### Data 

We will continue to work with the [Parking Violations Issued - Fiscal Year 2022](https://data.cityofnewyork.us/City-Government/Parking-Violations-Issued-Fiscal-Year-2022/7mxj-7a6y) 
dataset from NYC Open Data.

## Let's run some pandas code 

In [ ]:
import pandas as pd

In [ ]:
# read some columns of the dataset
df = pd.read_parquet(
    "../data/nyc_parking_violations_2022.parquet",
    columns=[
        "Registration State",
        "Violation Code",
        "Vehicle Body Type",
        "Vehicle Make",
        "Violation Time",
        "Violation County",
        "Vehicle Year",
        "Violation Description",
        "Issue Date",
        "Summons Number",
    ],
)

# view a random sample of 10 rows:
df.head()

## Parking violations by Registration state 

Each record in our dataset contains the state of registration of the offending vehicle, and the type of parking violation. 
To get the most common type of violation for vehicles registered in different states, we use [value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) and [GroupBy.head](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.head.html):

In [ ]:
%%time

(
    df[["Registration State", "Violation Description"]]  # get only these two columns
    .value_counts()  # get the count of violations per state and per type of offence
    .groupby("Registration State")  # group by state
    .head(
        1
    )  # get the first row in each group (the type of violation with the largest count)
    .sort_index()  # sort by state name
    .reset_index()
)

The code above uses [method chaining](https://tomaugspurger.net/posts/method-chaining/) to combine a series of operations 
into a single statement. You might find it useful to break the code up into multiple statements and inspect each of the intermediate results.

## What types of vehicle are most frequently involved in parking violations?

In [ ]:
%%time

(
    df.groupby(["Vehicle Body Type"])
    .agg({"Summons Number": "count"})
    .rename(columns={"Summons Number": "Count"})
    .sort_values(["Count"], ascending=False)
)

From the [Vehicle Body Type dictionary](https://data.ny.gov/api/assets/83055271-29A6-4ED4-9374-E159F30DB5AE) form the 
NYC Parking Data.

- SUBN: SUBURBAN
- 4DSD: FOUR-DOOR SEDAN
- VAN: VAN TRUCK
- DELV: DELIVERY TRUCK
- PICK: PICK-UP TRUCK

**Exercise:** Get the top 5 parking offenders by Vehicle Brands. 

<details>
  <summary>Solution (click dropdown) </summary>
  <p>

```python
# to run this type it in a code cell
(df
 .groupby(["Vehicle Make"])
 .agg({"Summons Number": "count"})
 .rename(columns={"Summons Number": "Count"})
 .sort_values(["Count"], ascending=False)
 .head(5)
)
```
  </p>
</details>


In [ ]:
## your solution here

## Day of the week when more parking violations occur

In [ ]:
%%time
weekday_names = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday",
}

df["Issue Date"] = df["Issue Date"].astype("datetime64[ms]")
df["issue_weekday"] = df["Issue Date"].dt.weekday.map(weekday_names)

df.groupby(["issue_weekday"])["Summons Number"].count().sort_values(ascending=False)

## What is the county where most of the parking violations happen? 

In [ ]:
%%time
(df.groupby("Violation County").size().sort_values(ascending=False).head(10))

# Let's accelerate

If you have nvdashboard installed, select the `cudf.pandas` in the GPU accel toggle.  

**Note**: if you are running this in a system where you don't have nvdashbaord, use `%load_ext cudf.pandas` as instructed above.

**Exercise:** Find the top 5 most common parking violations for vehicles that are either SUVs (Vehicle Body Type = "SUBN")
or pickup trucks (Vehicle Body Type = "PICK"), but only for vehicles made after 2010, and show the count for each violation type.

<details>
  <summary>Solution (click dropdown) </summary>
  <p>

```python
# to run this type it in a code cell

# Filter for SUVs and pickup trucks made after 2010
recent_suv_pickup = df[
    (df["Vehicle Body Type"].isin(["SUBN", "PICK"])) & 
    (df["Vehicle Year"] > 2010)
]

# Group by violation type and count, then get top 5
(
    recent_suv_pickup
    .groupby("Violation Description")
    .size()
    .sort_values(ascending=False)
    .head(5)
    .rename("Number of Violations")
)
```
  </p>
</details>

In [ ]:
# your solution here

## FAQ

### When should I use cuDF (direct import) versus cudf.pandas?

**Use cudf.pandas if**
- You have existing pandas code and you want to run it on GPUs with 0 effort
- The ability to run the same code on GPU-enabled as well as CPU-only systems is important

**Use cuDF (direct import) if:**
- You want everything to run on GPU (CPU fallback is prohibitively expensive)
- You need functionality that cuDF provides but pandas does not

### How do you ensure pandas compatibility?

- We run the entire pandas unit test suite with cudf.pandas enabled
    -  ~94% of the tests passing – a few minor differences
- We turn on cuDF’s “pandas compatibility mode” (ensures result ordering matches pandas, etc.)

    ```python
    cudf.set_option("mode.pandas_compatible", True)
    ```

## Tips and Tricks

- Use the profiler to learn which function are run on CPU and GPU (doesn't report CPU<->GPU transfer)
- CPU fallback involves copying data between CPU and GPU – twice in the worst case.
- Use GPU-supported operations as much as possible
- GPU memory is limited compared to CPU RAM
    - If you ran out of GPU memory, it will fall back to CPU (**unexpected slowdown**)
    - Keep only the data that you need 
    - Monitor GPU usage (only on Jupyter - NVDashboard)
- When possible use idiomatic pandas and avoid udfs  
